<a href="https://colab.research.google.com/github/aszczi/Urban_mobility_in_Cracow/blob/main/Mobilno%C5%9B%C4%87_heatmap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Natężenie ruchu (Heatmap) - Historia Traffic

Ten notatnik zajmuje się tworzeniem map opóźnień ("heatmap" oraz interaktywnych punktów) w oparciu o informacje z pliku `traffic_history.csv`. Analizujemy w nim zagęszczenie i średnie opóźnienia uliczne z podziałem na godziny.

In [ ]:
# Instalacja niezbędnych bibliotek, upewnij się, że używasz odpowiedniego środowiska python.
!pip install pandas plotly osmnx matplotlib folium scipy

In [ ]:
import pandas as pd
import plotly.express as px
import osmnx as ox
import matplotlib.pyplot as plt

# 1. Wczytanie i przygotowanie danych samochodowych o ruchu
df = pd.read_csv("https://raw.githubusercontent.com/aszczi/Urban_mobility_in_Cracow/refs/heads/main/traffic_history.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.strftime('%Y-%m-%d %H:00')

# Pogrupowanie danych z perspektywy krzyżówek/ulic i godzin
df_grouped = df.groupby(['hour', 'point_name', 'lat', 'lon'], as_index=False).agg({
    'delay_sec': 'mean',
    'congestion_index_pct': 'mean',
    'current_speed_kmph': 'mean'
})

df_grouped = df_grouped.sort_values(by=['hour'])
df_grouped.head()

## Dynamiczna mapa z nałożeniem na układ ulic (routing)

Zgodnie z poleceniem, całkowicie rezygnujemy ze statycznej mapy punktowej. Opóźnienia nakładamy symulując "miejskie trasy" poprzez połączenie zgłoszonych punktów pomiarowych ("przystanków") na siatce dróg. 

Do animacji na mapie bazowej zaprzężemy wtyczkę `folium.plugins.TimestampedGeoJson`, natomiast rzeczywistą wytyczoną po krawędziach trasę pomiędzy punktami wygeneruje poszukiwanie `ox.shortest_path` z pakietu OSMnx. Narysowane w ten sposób linie będą podświetlane niczym natężenie ruchu z nawigacji Google Maps.

In [ ]:
import folium
from folium.plugins import TimestampedGeoJson
import osmnx as ox
import networkx as nx
import matplotlib.colors as mcolors
import pandas as pd

# 1. Wczytanie grafu drogowego dla miasta z OSM
lokalizacja = "Kraków, Poland"
print(f"Pobieranie grafu miasta dla {lokalizacja}... ")
G = ox.graph_from_place(lokalizacja, network_type="drive", simplify=True)

# 2. Pobieramy unikalne lokalizacje punktów/przystanków i wiążemy je z węzłami skrzyżowań
unique_points = df_grouped.drop_duplicates(subset=['point_name']).copy()

lats = unique_points['lat'].values
lons = unique_points['lon'].values

print("Wyliczanie najbliższych ulic i wyznaczanie tras (odcinków)...")
try:
    nodes = ox.nearest_nodes(G, X=lons, Y=lats)
except AttributeError:
    # Kompatybilność z minimalnie starszym osmnx
    nodes = ox.distance.nearest_nodes(G, X=lons, Y=lats)
    
unique_points['osmid'] = nodes

# 3. Budujemy trasy (łączymy węzły chronologicznie w trasy by imitowały przebieg linii)
nodes_list = unique_points['osmid'].tolist()
point_names = unique_points['point_name'].tolist()

route_segments = []
for i in range(len(nodes_list) - 1):
    source = nodes_list[i]
    target = nodes_list[i+1]
    try:
        # Wyszukujemy ciągłość jezdni pomiędzy opisywanymi miejscami (krótkie nawigowanie)
        path = nx.shortest_path(G, source, target, weight='length')
        route_segments.append({
            'p_from': point_names[i], 
            'p_to': point_names[i+1],
            'path': path
        })
    except nx.NetworkXNoPath:
        continue # W razie dziury w siatce gubimy odcinek (niektóre ulice jednokierunkowe mogą to spowodować)

# 4. Generowanie features pod wtyczkę TimestampedGeoJson
features = []
cmap = plt.get_cmap('autumn_r') # Czerwony najgorszy, pomarańczowy, żółty najlepszy
norm = mcolors.Normalize(vmin=0, vmax=100)

# Upewniamy się, że format czasu jest folium-friendly (ISO 8601)
df_grouped['timestamp_folium'] = pd.to_datetime(df_grouped['hour']).dt.strftime('%Y-%m-%dT%H:00:00')
unique_hours = df_grouped['timestamp_folium'].unique()

print("Przetwarzanie opóźnień w przedziały godzinowe...")
for hour in unique_hours:
    df_hour = df_grouped[df_grouped['timestamp_folium'] == hour]
    
    for segment in route_segments:
        p_from = segment['p_from']
        p_to = segment['p_to']
        
        # Uśredniamy natężenie między początkiem a końcem "trasy"
        val_from = df_hour[df_hour['point_name'] == p_from]['congestion_index_pct']
        val_to = df_hour[df_hour['point_name'] == p_to]['congestion_index_pct']
        
        v1 = val_from.values[0] if not val_from.empty else 0
        v2 = val_to.values[0] if not val_to.empty else 0
        
        if v1 == 0 and v2 == 0:
            continue
            
        avg_congestion = (v1 + v2) / 2

        # Zamiana węzłów drogi nawigacji na koordynaty geograficzne w formacie LineString [lon, lat]
        coords = [[G.nodes[n]['x'], G.nodes[n]['y']] for n in segment['path']]
        
        features.append({
            "type": "Feature",
            "geometry": {
                "type": "LineString",
                "coordinates": coords
            },
            "properties": {
                "times": [hour] * len(coords), # Pasek osi czasu obejmie równomiernie krawędzie linii
                "style": {
                    "color": mcolors.to_hex(cmap(norm(avg_congestion))),
                    "weight": 6,
                    "opacity": 0.85
                }
            }
        })

# 5. Generujemy docelową animowaną mapę
print("Budowanie widoku mapy...")
folium_map = folium.Map(location=[50.0647, 19.9450], zoom_start=13, tiles="cartodbdark_matter")

TimestampedGeoJson(
    {"type": "FeatureCollection", "features": features},
    period="PT1H",
    add_last_point=False,
    auto_play=True,
    loop=True,
    max_speed=1,
    loop_button=True,
    time_slider_drag_update=True
).add_to(folium_map)

# Wywołujemy widok mapy jako wyjście w kafelku notatnika
folium_map